# Section 6.2: ablation of the meta-prior

Two networks with the same architecture and training length (80k steps) that differ **only in the
meta-prior**:

| Network | Meta-prior | Preset |
|---|---|---|
| full | the meta-prior of the thesis | `small` → `runs/small` |
| ablated | no interaction-only family, no weak-effect regime, every interacting variable of the pairwise family keeps its main effect | `ablation` → `runs/ablation` |

Design C: $y = x_1x_2 + \sin(x_3x_4) + \mathbb{1}\{x_5>0\}x_6 + \varepsilon$, $p = 100$, Toeplitz Gaussian
predictors with $\rho \in \{0.3, 0.7\}$, SNR $\in \{1, 5\}$, $n \in \{100, 300, 800\}$, 100 replicates
(1200 data sets). DART is run at $n = 100$ only (400 data sets).

Runtime: a GPU (A100 or L4). Each training takes about 4 hours on an A100, runs in the background and
resumes from its last checkpoint if restarted; DART takes about an hour on the CPU cores, after training
(training uses every core to simulate data).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, subprocess, collections
# folder of this repository on your Google Drive (upload it there, or clone it with git)
CODE = '/content/drive/MyDrive/pfn-bayesian-variable-selection'
assert os.path.exists(f'{CODE}/bench/run.py'), f'repository not found in {CODE}'
os.chdir(CODE)
os.makedirs('results/logs', exist_ok=True)
!pip -q install tabulate
!nvidia-smi --query-gpu=name --format=csv,noheader
NCPU = os.cpu_count(); print('CPU cores:', NCPU)
ENV = {**os.environ, 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1'}

def launch(args, log):
    """Start `python -m <args>` in the background, with its log in results/logs."""
    return subprocess.Popen(['python', '-W', 'ignore', '-m'] + args, env=ENV, start_new_session=True,
                            stdout=open(f'results/logs/{log}.log', 'a'), stderr=subprocess.STDOUT)

def count(folder):
    c, e = collections.Counter(), collections.Counter()
    for f in glob.glob(f'{folder}/*.jsonl'):
        for line in open(f):
            if line.strip():
                r = json.loads(line)
                (e if r.get('error') else c)[r['method']] += 1
    return c, e

## 2. Train the two networks (in the background)

The full network is skipped if it is already there (e.g. from the notebook of Section 6.1). Rerun cell 3 to follow the training.

In [ ]:
if os.path.exists('runs/small/final.pt'):
    print('runs/small/final.pt already there')
else:
    train = launch(['pfn.train', '--preset', 'small', '--out', 'runs/small'], 'train_small')
    print('training started; follow it with: !tail -n 2 runs/small/log.csv')

if os.path.exists('runs/ablation/final.pt'):
    print('runs/ablation/final.pt already there')
else:
    train_abl = launch(['pfn.train', '--preset', 'ablation', '--out', 'runs/ablation'], 'train_ablation')
    print('training of the ablated network started')

## 3. Training progress (rerun whenever you like)

In [ ]:
for run in ('runs/small', 'runs/ablation'):
    if os.path.exists(f'{run}/final.pt'):
        print(run, 'finished')
    elif os.path.exists(f'{run}/log.csv'):
        print(run, open(f'{run}/log.csv').read().strip().splitlines()[-1])
    else:
        print(run, 'no log yet')

## 4. Evaluate the two networks on design C

In [ ]:
SP, ABL = 'runs/small/final.pt', 'runs/ablation/final.pt'
assert os.path.exists(SP) and os.path.exists(ABL), 'the networks are not trained yet'
!python -W ignore -m bench.run --study ablation --reps 100 --methods selectpfn --sp full=$SP --sp ablated=$ABL --out results/ablation

## 5. R and the BART package (for DART)

In [ ]:
if shutil.which('Rscript') is None:
    !apt-get -qq install -y r-base > /dev/null
!Rscript -e 'if (!requireNamespace("BART", quietly=TRUE)) install.packages("BART", repos="https://cloud.r-project.org", quiet=TRUE); library(BART); cat("BART ok\n")'

## 6. DART on design C, n = 100 (in the background, after training)

No GPU needed: this part can also run in a CPU-only session (cells 1, 5 and 6).

In [ ]:
K = NCPU
procs = [launch(['bench.run', '--study', 'c_simple', '--reps', '100', '--methods', 'dart', '--device', 'cpu',
                 '--out', 'results/c_dart', '--shard', str(i), '--nshards', str(K)], f'c_dart_{i}')
         for i in range(K)]
print(len(procs), 'DART processes started')

## 7. Progress (rerun whenever you like)

In [ ]:
c1, e1 = count('results/ablation'); c2, e2 = count('results/c_dart')
for m, tot, c, e in (('selectpfn:full', 1200, c1, e1), ('selectpfn:ablated', 1200, c1, e1), ('dart', 400, c2, e2)):
    print(f'{m:18s} {c[m]:5d} / {tot}   errors: {e[m]}')

## 8. Tables and Figure 6.2

Works on the data sets done so far.

In [ ]:
!python -W ignore -m bench.ablation_report --results results/ablation,results/c_dart --save results/ablation_report.md --fig results/figures > /dev/null
!head -30 results/ablation_report.md
from IPython.display import Image, display
display(Image('results/figures/ablation_detection.png', width=900))